# Auditable Training

Pre-registration, watchdogs, forensics, and atomic artifacts are first-class APIs.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from parallelcbf.api import PreRegistrationSpec
from parallelcbf.ops import DefaultWatchdogRegistry, FailureForensics, JsonPreRegistration, ThresholdWatchdog

with TemporaryDirectory() as tmp:
    root = Path(tmp)
    prereg = JsonPreRegistration()
    prereg.add_spec(PreRegistrationSpec(name="zero_collision", hypothesis="No hard violations.", metric_name="h_hard_violation_rate", threshold=0.0, comparison="eq", sample_size=100))
    commit = prereg.commit_to_artifact(root / "prereg.json")

    registry = DefaultWatchdogRegistry()
    registry.register(ThresholdWatchdog("h_hard_violation_rate", 0.0))
    events = registry.update({"h_hard_violation_rate": 0.01}, step=1)

    forensics = FailureForensics(capacity=8)
    forensics.push(step=1, metrics={"h_hard_violation_rate": 0.01})
    dump_path = forensics.dump_to_disk(reason="demo", path=root)
    print({"sha256": commit.sha256[:8], "events": len(events), "dump": dump_path.name})